In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [ ]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()

    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()

    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [ ]:
"""
Determine investment windows where the available balance exceeds a threshold and track the duration of these windows.
df: DataFrame with 'TransactionDate' and 'Available' columns
threshold = 1,000,000 - minimum balance to consider
Returns the investment windows as a DataFrame with columns:
    'start_date': Start date of the investment window
    'end_date': End date of the investment window (None if ongoing)
    'amount': The balance amount during the window
    'duration': Duration of the window in days
"""
# Function to detect window intervals
def find_window_intervals(df, threshold=1_000_000):
    df = df.sort_values('TransactionDate').reset_index(drop=True)
    intervals = []

    for i in range(len(df)):
        base_row = df.iloc[i]
        base_amount = base_row['Available']
        if base_amount < threshold:
            continue

        start_date = base_row['TransactionDate']
        duration = 1
        end_date = None

        for j in range(i + 1, len(df)):
            if df.iloc[j]['Available'] >= base_amount:
                duration += 1
            else:
                end_date = df.iloc[j]['TransactionDate']
                break
        else:
            end_date = df.iloc[-1]['TransactionDate']

        if not intervals or intervals[-1]['start_date'] != start_date:
            intervals.append({
                'start_date': start_date,
                'end_date': end_date,
                'amount': base_amount,
                'duration': duration
            })

    # Create DataFrame
    df_intervals = pd.DataFrame(intervals)

    # Format columns
    df_intervals['start_date'] = pd.to_datetime(df_intervals['start_date']).dt.date
    df_intervals['end_date'] = pd.to_datetime(df_intervals['end_date']).dt.date
    df_intervals['amount'] = df_intervals['amount'].round(2)
    df_intervals['change'] = (df_intervals['amount'] - df_intervals['amount'].shift(1).fillna(0)).round(2)

    return df_intervals


In [ ]:
# Step 2: Keep Only the Max Amount Interval for Each Overlapping Period
def drop_overlapping_lower_intervals(df):
    # Group by start_date and end_date, then get the row with maximum available amount
    result = df.loc[df.groupby(['start_date', 'end_date'])['amount'].idxmax()]
    return result

In [ ]:
# Step 3: Update Changed Amounts Based on Prior Date from the Original Data
def update_changed_amounts(df_intervals, df_data):

    df_intervals['start_date'] = pd.to_datetime(df_intervals['start_date'])
    df_intervals['previous_date'] = df_intervals['start_date'] - pd.Timedelta(days=1)

    df_data['TransactionDate'] = pd.to_datetime(df_data['TransactionDate'])
    df_temp = df_data[['TransactionDate', 'Available']].copy()

    df_intervals = df_intervals.merge(
        df_temp,
        left_on='previous_date',
        right_on='TransactionDate',
        how='left'
    )
    df_intervals = df_intervals.rename(columns={'Available': 'previous_amount'})

    # logging.warning("Some previous amounts are NaN. There is no previous date in the original data for the first record.")
    df_intervals.loc[0, 'previous_amount'] = 0

    df_intervals['change'] = (df_intervals['amount'] - df_intervals['previous_amount']).round(2)
    # Replace -0.0 with 0.0
    df_intervals['change'] = df_intervals['change'].apply(lambda x: 0.0 if x == 0 else x)

    # Cleanup (optional)
    df_intervals = df_intervals.drop(columns=['TransactionDate'])
    return df_intervals

In [ ]:
# MAIN
#     analyze_balance_data() for running the asset class balances processing

# In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 5
# Load and print the data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-12-31')

with pd.option_context('display.max_columns', None,
                       'display.max_rows', None,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
display(data.head(30))

data.info()

all_intervals = find_window_intervals(data).sort_values(['start_date'], ascending=[True])
intervals = drop_overlapping_lower_intervals(all_intervals)

final_intervals = update_changed_amounts(intervals, data)

# Debug display
print("\nInvestment windows:", len(intervals))
with pd.option_context('display.max_columns', None,
                       'display.max_rows', 100,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
display(final_intervals.head(25))


